# Step 2 (v2): Retrain with HealthVer's own train split
v1 (trained on PUBHEALTH + SciFact only) scored macro-F1 **0.47** on HealthVer. Too low.
v2 adds HealthVer's `train` split. Its `test` split stays completely untouched until the final evaluation.

**Plan**
1. Load `unified_v2.parquet` from step 1 (v2).
2. Train on PUBHEALTH-train + SciFact-train (×3) + HealthVer-train. Drop any HealthVer-train row whose claim text also appears in HealthVer val/test.
3. Weight the loss so the rare `NEI` class still matters.
4. Pick the best epoch by macro-F1 on PUBHEALTH-val + SciFact-val + HealthVer-val.
5. Evaluate once on PUBHEALTH-test and **HealthVer-test**.
6. Compare v1 and v2 on the *same* HealthVer-test rows (needs v1 saved on Drive from the first run).
7. Save to `verdict_model_v2/` and `logits_v2.npz`. v1 files are left alone.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
%pip install -q "transformers>=4.46,<5" accelerate sentencepiece protobuf scikit-learn

In [ ]:
import os, shutil, random, numpy as np, pandas as pd, torch, torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

# ---- config ----
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 384          # abstracts are ~300 tokens; shorter = faster
EPOCHS = 3
LR = 2e-5
BATCH = 16
SCIFACT_REPEAT = 3     # oversample the small, in-domain dataset
SEED = 42
SMOKE = False          # True = tiny quick run just to check the code works
# ----------------
LABELS = ["SUPPORTS", "REFUTES", "NEI"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not SMOKE:
    assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (smoke test)")

## Google Drive (so a disconnect doesn't cost you the model)
Checkpoints, the final model and the scores are written straight to Drive. **Approve the Drive pop-up** when it appears.
If the runtime dies mid-training, reconnect, rerun all cells, and training resumes from the last saved epoch.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:          # not on Colab (local test only)
    DRIVE = "veritas_local"
os.makedirs(DRIVE, exist_ok=True)
CKPT_DIR = f"{DRIVE}/ckpt_v2"
print("saving to:", DRIVE)

## Load data
Looks for `unified_v2.parquet` here, then in Drive, otherwise asks you to upload it from step 1.

In [ ]:
if not os.path.exists("unified_v2.parquet"):
    if os.path.exists(f"{DRIVE}/unified_v2.parquet"):
        shutil.copy(f"{DRIVE}/unified_v2.parquet", "unified_v2.parquet")
    else:
        from google.colab import files
        files.upload()      # choose unified_v2.parquet from step 1
if not os.path.exists(f"{DRIVE}/unified_v2.parquet"):
    shutil.copy("unified_v2.parquet", f"{DRIVE}/unified_v2.parquet")   # keep a copy on Drive
df = pd.read_parquet("unified_v2.parquet")

get = lambda src, sp: df[(df.source == src) & (df.split == sp)].reset_index(drop=True)

hv_train, hv_val, hv_test = get("healthver", "train"), get("healthver", "validation"), get("healthver", "test")
seen = set(hv_val.claim) | set(hv_test.claim)
n_before = len(hv_train)
hv_train = hv_train[~hv_train.claim.isin(seen)]        # no claim text shared with val/test
print("HealthVer train rows dropped for sharing a claim with val/test:", n_before - len(hv_train))

train_df = pd.concat([
    get("pubhealth", "train"),
    pd.concat([get("scifact", "train")] * SCIFACT_REPEAT),
    hv_train,
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

ph_val, sf_val, ph_test = get("pubhealth", "validation"), get("scifact", "validation"), get("pubhealth", "test")

if SMOKE:
    train_df = train_df.sample(120, random_state=0).reset_index(drop=True)
    ph_val, sf_val, hv_val, ph_test, hv_test = [x.head(40) for x in (ph_val, sf_val, hv_val, ph_test, hv_test)]

val_df = pd.concat([ph_val, sf_val, hv_val]).reset_index(drop=True)
print({k: len(v) for k, v in dict(train=train_df, val=val_df, ph_test=ph_test, hv_test=hv_test).items()})
print(pd.crosstab(train_df.source, train_df.label_name))
print(train_df.label_name.value_counts())

## Class weights
`NEI` is rare, so a plain loss lets the model ignore it. Weight = (N / (3 × class count)) ^ 0.5.
The square root keeps the boost gentle. Full inverse-frequency weights tend to make the model over-predict the rare class.

In [ ]:
counts = train_df.label.value_counts().reindex([0, 1, 2]).values
cw = torch.tensor((len(train_df) / (3 * counts)) ** 0.5, dtype=torch.float)
print(dict(zip(LABELS, cw.numpy().round(2))))

## Tokenize
Each example becomes `[CLS] claim [SEP] evidence [SEP]`. `truncation="only_second"` cuts the evidence, never the claim.
Padding is done per batch (dynamic), so short PUBHEALTH rows are cheap.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(d):
    ds = Dataset.from_pandas(d[["claim", "evidence", "label"]].rename(columns={"label": "labels"}),
                             preserve_index=False)
    return ds.map(lambda b: tok(b["claim"], b["evidence"], truncation="only_second", max_length=MAX_LEN),
                  batched=True, remove_columns=["claim", "evidence"])

train_ds, val_ds = make_ds(train_df), make_ds(val_df)
print(train_ds)

## Model + weighted trainer
Only the loss is customised: cross-entropy with the class weights above.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = F.cross_entropy(out.logits.float(), labels,
                               weight=self.class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "acc": float((preds == p.label_ids).mean())}

args = TrainingArguments(
    output_dir=CKPT_DIR, num_train_epochs=1 if SMOKE else EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
    warmup_ratio=0.1, weight_decay=0.01, fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    group_by_length=True, logging_steps=50, report_to="none", seed=SEED)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                          class_weights=cw)

## Train (about 30–60 min on a T4)
Resumes automatically if a checkpoint already exists on Drive.

In [ ]:
resume = os.path.isdir(CKPT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(CKPT_DIR))
print("resuming from checkpoint" if resume else "starting fresh")
trainer.train(resume_from_checkpoint=True if resume else None)

## Evaluate
Shown per dataset so you can see where it is weak. **HealthVer TEST is the honest number**: it was never used for training or for picking the epoch.

In [ ]:
def report(name, d):
    out = trainer.predict(make_ds(d))
    y, pred = out.label_ids, out.predictions.argmax(-1)
    print(f"\n===== {name}  (n={len(d)})  macro-F1 = {f1_score(y, pred, average='macro'):.3f}  acc = {(y == pred).mean():.3f}")
    print(classification_report(y, pred, labels=[0, 1, 2], target_names=LABELS, zero_division=0))
    print("confusion (rows = true, cols = pred):\n", confusion_matrix(y, pred, labels=[0, 1, 2]))
    return out.predictions, y

res = {
    "pubhealth_val":  report("PUBHEALTH val",  ph_val),
    "scifact_val":    report("SciFact val",    sf_val),
    "healthver_val":  report("HealthVer val",  hv_val),
    "pubhealth_test": report("PUBHEALTH test", ph_test),
    "healthver_test": report("HealthVer TEST (never touched)", hv_test),
}

## Save (straight to Drive)
The scores go to `logits_v2.npz` for step 5 (calibration + abstention). The model goes to `verdict_model_v2/`.

In [ ]:
np.savez(f"{DRIVE}/logits_v2.npz", **{f"{k}_logits": v[0] for k, v in res.items()},
                                **{f"{k}_labels": v[1] for k, v in res.items()})
trainer.save_model(f"{DRIVE}/verdict_model_v2"); tok.save_pretrained(f"{DRIVE}/verdict_model_v2")
print(sorted(os.listdir(DRIVE)))

## Before vs after, on the same HealthVer test rows
Loads your first model (`verdict_model/` on Drive) and scores it on exactly the same test rows as v2.

In [ ]:
V1_DIR = f"{DRIVE}/verdict_model"
if not os.path.isdir(V1_DIR):
    print("v1 model not found at", V1_DIR, "- skipping comparison")
else:
    v1 = AutoModelForSequenceClassification.from_pretrained(V1_DIR)
    t1 = Trainer(model=v1, data_collator=DataCollatorWithPadding(tok),
                 args=TrainingArguments(output_dir="tmp_v1", per_device_eval_batch_size=32,
                                        fp16=torch.cuda.is_available(), report_to="none"))
    out1 = t1.predict(make_ds(hv_test))
    y = out1.label_ids
    p1, p2 = out1.predictions.argmax(-1), res["healthver_test"][0].argmax(-1)
    print(f"HealthVer TEST (n={len(y)})")
    print(f"{'model':32s} macro-F1   acc    F1[SUP]  F1[REF]  F1[NEI]")
    for name, p in [("v1 (no HealthVer in training)", p1), ("v2 (+ HealthVer train)", p2)]:
        f = f1_score(y, p, average=None, labels=[0, 1, 2])
        print(f"{name:32s} {f1_score(y, p, average='macro'):.3f}     {(y == p).mean():.3f}  {f[0]:.3f}    {f[1]:.3f}    {f[2]:.3f}")

## What to check before Step 3
1. **HealthVer TEST: v2 vs v1.** This before/after pair is a headline result for your pitch.
2. **PUBHEALTH and SciFact**: did they drop compared with v1 (0.73 and 0.71)? A small drop is fine.
3. **Remaining weakness**: look at the HealthVer-test confusion matrix. If SUPPORTS is still mostly predicted as REFUTES, PUBHEALTH's explanation-style shortcuts may still be hurting. A fix to try: train without PUBHEALTH, or repeat SciFact/HealthVer more.
4. Loss going down but val macro-F1 flat after epoch 1 means overfitting. Use fewer epochs.